In [ ]:
import sys
import os
import zipfile
import random
import pandas as pd
import librosa
import torchaudio
import torch
import matplotlib.pyplot as plt

from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
SECRETS_DIR = PROJECT_DIR / "secrets"
DATA_DIR = PROJECT_DIR / "data"
DATA_ZIP_DIR = DATA_DIR / "zip"
DATA_RAW_DIR = DATA_DIR / "raw"
DATA_PREPROCESSED_DIR = DATA_DIR / "processed"
sys.path.append(str(PROJECT_DIR))
print(PROJECT_DIR)

In [ ]:
from src.core.config import settings

REQUIRES_DOWNLOAD = False

In [ ]:
if REQUIRES_DOWNLOAD:
    from src.services.google_drive import GoogleDriveService

    gd_service = GoogleDriveService(str(SECRETS_DIR / "google_credentials.json"))

    files = gd_service.list_files(folder_path=settings.GDRIVE_ROOT_PATH)
    for file in files:
        gd_service.download_file(
            local_path=DATA_ZIP_DIR / file["name"],
            file_id=file["id"],
        )

In [ ]:
# YOLO Aliases
from typing import TypeAlias
import numpy as np
import numpy.typing as npt

# NumPy-based aliases
ImageArray: TypeAlias = npt.NDArray[np.uint8]  # HxWxC image
BoxesArray: TypeAlias = npt.NDArray[np.float32]  # [N,4] (x1,y1,x2,y2) or (x,y,w,h)
ScoresArray: TypeAlias = npt.NDArray[np.float32]  # [N]
ClassIdsArray: TypeAlias = npt.NDArray[np.int32]  # [N]

# Standard-library aliases
Point: TypeAlias = tuple[float, float]
BoundingBox: TypeAlias = tuple[
    float, float, float, float
]  # [x, y, sqrt(W), sqrt(H), C (confidence))]
Detection: TypeAlias = tuple[BoundingBox, int, float]  # (bbox, class_id, score)
Detections: TypeAlias = list[Detection]

In [ ]:
def uncompress_zip(zip_path: Path, extract_to: Path) -> None:
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)


def get_annotation_file(annotations_dir: Path, record_file: Path) -> Path | None:
    annotation_file = record_file.stem + ".txt"
    return_path = annotations_dir / annotation_file
    return return_path if return_path.exists() else None


def load_annotations(annotation_file: Path) -> pd.DataFrame:
    return pd.read_csv(annotation_file, sep="\t")


def load_audio(
    audio_file: Path, sample_rate: int | float | None = None
) -> tuple[npt.NDArray[np.float32], int | float]:
    return librosa.load(audio_file, sr=sample_rate)


def clip_audio(
    waveform: npt.NDArray[np.float32] | torch.Tensor,
    limit_sec: float,
    offset_sec: float,
    sample_rate: int | float,
) -> npt.NDArray[np.float32] | torch.Tensor:
    start_sample = int(offset_sec * sample_rate)
    end_sample = int((offset_sec + limit_sec) * sample_rate)
    return waveform[start_sample:end_sample]

In [ ]:
for zip_file in os.listdir(DATA_ZIP_DIR):
    if zip_file.endswith(".zip"):
        basename = Path(zip_file).stem
        zip_path = DATA_ZIP_DIR / zip_file
        extract_to = DATA_RAW_DIR / basename
        uncompress_zip(zip_path, extract_to)

In [ ]:
SAMPLE_DIR = DATA_RAW_DIR / random.choice(os.listdir(DATA_RAW_DIR))
SAMPLE_DIR

In [ ]:
from torchaudio.transforms import Spectrogram

NFFT = 2048
HOP_LENGTH = 256
spec = Spectrogram(n_fft=NFFT, hop_length=HOP_LENGTH)

records_files: list[Path] = [
    SAMPLE_DIR / wav_file
    for wav_file in os.listdir(SAMPLE_DIR)
    if wav_file.endswith(".wav")
]
sample_record: Path = random.choice(records_files)
sample_annotation: Path | None = get_annotation_file(SAMPLE_DIR, sample_record)
annotations: pd.DataFrame | None = (
    load_annotations(sample_annotation) if sample_annotation else None
)

audio, sr = load_audio(sample_record, sample_rate=44100)
clip_duration_sec = 5.0
offset_sec = 0.0
cliped = clip_audio(audio, clip_duration_sec, offset_sec, sr)
spectrogram = spec(torch.from_numpy(cliped)).numpy()
plt.figure(figsize=(16, 8))
plt.imshow(10 * np.log10(spectrogram + 1e-10), aspect="auto", origin="lower")
plt.title(f"Spectrogram {sample_record.name}")
plt.show()

In [ ]:
print(f"Audio Shape: {audio.shape}, Sample Rate: {sr}")
print(
    f"Spectrogram Shape: {spectrogram.shape} == {(NFFT // 2 + 1, int(np.ceil(len(cliped) / HOP_LENGTH)))}"
)

ANNOTATION_COLUMNS = {
    "Selection": "int64",
    "View": "str",
    "Channel": "int64",
    "Begin Time (s)": "float64",
    "End Time (s)": "float64",
    "Low Freq (Hz)": "float64",
    "High Freq (Hz)": "float64",
    "Inband Power (dB FS)": "float64",
    "Species": "str",
    "Call type": "str",
    "Rating": "str",
    "Reference": "float64",
}

annotations

In [ ]:
COORDINATE = tuple[int, int, int, int, str, str]  # (x1, y1, x2, y2, species, call_type)


def get_global_cords(
    sr: int | float, nfft: int, hop_length: int, annotations: pd.DataFrame
) -> list[COORDINATE]:
    global_cords = []
    for _, row in annotations.iterrows():
        start_time = row["Begin Time (s)"]
        end_time = row["End Time (s)"]
        low_freq = row["Low Freq (Hz)"]
        high_freq = row["High Freq (Hz)"]

        x1 = int(start_time * sr / hop_length)
        x2 = int(end_time * sr / hop_length)
        y1 = int(low_freq * nfft / sr)
        y2 = int(high_freq * nfft / sr)

        species = row["Species"]
        call_type = row["Call type"]
        global_cords.append((x1, y1, x2, y2, species, call_type))
    return global_cords

In [ ]:
def global_cords_to_yolo(
    global_cords: list[COORDINATE], img_w: int, img_h: int, class_to_id: dict[str, int]
) -> list[tuple[int, float, float, float, float]]:
    yolo_labels: list[tuple[int, float, float, float, float]] = []
    for x1, y1, x2, y2, species, call_type in global_cords:
        # opcional: clase por especie+tipo
        class_key = f"{species}__{call_type}"
        if class_key not in class_to_id:
            continue

        # clip por seguridad
        x1 = max(0, min(x1, img_w - 1))
        x2 = max(0, min(x2, img_w - 1))
        y1 = max(0, min(y1, img_h - 1))
        y2 = max(0, min(y2, img_h - 1))
        if x2 <= x1 or y2 <= y1:
            continue

        xc = ((x1 + x2) / 2) / img_w
        yc = ((y1 + y2) / 2) / img_h
        w = (x2 - x1) / img_w
        h = (y2 - y1) / img_h

        yolo_labels.append((class_to_id[class_key], xc, yc, w, h))
    return yolo_labels

In [ ]:
global_cords = (
    get_global_cords(sr, NFFT, HOP_LENGTH, annotations)
    if annotations is not None
    else []
)
global_cords

In [ ]:
class_mapping = (
    {f"{row['Species']}__{row['Call type']}": 0 for idx, row in annotations.iterrows()}
    if annotations is not None
    else {}
)

yolo_cords = global_cords_to_yolo(
    global_cords,
    img_w=spectrogram.shape[1],
    img_h=spectrogram.shape[0],
    class_to_id=class_mapping,
)
yolo_cords

In [ ]:
# Try rendering the bounding boxes on the spectrogram
from matplotlib.patches import Rectangle

spectrogram = spec(torch.from_numpy(audio)).numpy()
plt.figure(figsize=(16, 8))
plt.imshow(10 * np.log10(spectrogram + 1e-10), aspect="auto", origin="lower")
plt.title(f"Spectrogram {sample_record.name}")
for x1, y1, x2, y2, species, call_type in global_cords:
    plt.gca().add_patch(
        Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            edgecolor="red",
            facecolor="none",
            linewidth=2,
        )
    )
plt.show()